## Execução local
Selecione o kernel **Python (FASE7)** e execute em ordem. Consulte `README_LOCAL.md` para instalação. Redes neurais usam PyTorch/CUDA; operações NumPy, áudio e modelos scikit-learn usam CPU.

In [ ]:
from pathlib import Path
import os, sys
# Localiza FASE7 tanto a partir do projeto quanto da pasta do notebook.
base = Path.cwd().resolve()
FASE7 = next((p for a in (base, *base.parents) for p in (a, a / "FASE7", a / "ANO1" / "FASE7") if p.name == "FASE7" and (p / "RNN").is_dir()), None)
if FASE7 is None:
    raise FileNotFoundError("Abra o Jupyter dentro da pasta FIAP ou FASE7.")
sys.path.insert(0, str(FASE7))
os.chdir(FASE7 / 'Cap 03')
print("Pasta de trabalho:", Path.cwd())


FIAP

# Aula 5 - Áudio: Extração de espectrogramas


In [ ]:
import librosa

# Carregar o arquivo
AUDIO_FILE = "audio_file.wav"

samples, sample_rate = librosa.load(AUDIO_FILE, sr=None) #SR: sample rate

In [ ]:
sample_rate

In [ ]:
# Plotar o áudio no aspectro do tempo
import matplotlib.pyplot as plt

plt.figure(figsize=(14,5))
librosa.display.waveshow(samples, sr=sample_rate)

In [ ]:
from IPython.display import Audio

Audio(AUDIO_FILE)

In [ ]:
import numpy as np
# Extrair o espectograma
sgram = librosa.stft(samples)
librosa.display.specshow(np.abs(sgram))

In [ ]:
# Valorizar o espectro percebido pelo humano
sgram_mag, _ = librosa.magphase(sgram)
mel_scale_sgram = librosa.feature.melspectrogram(S=sgram_mag,
                                                 sr=sample_rate)
librosa.display.specshow(mel_scale_sgram)

In [ ]:
# Converter para escala de decibéis
import numpy as np

mel_sgram = librosa.amplitude_to_db(mel_scale_sgram, ref=np.min)

librosa.display.specshow(mel_sgram, sr=sample_rate,
                         x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")

In [ ]:
# Ploto a imagem no formato consumível para CNN
librosa.display.specshow(mel_sgram, sr=sample_rate);

# Aula 5 - Trabalhando com biblioteca gtts

In [ ]:
print("Dependências: veja README_LOCAL.md. Não é necessário apt no Windows.")

In [ ]:
# Utilização padrão da biblioteca
import gtts as g

voz = g.gTTS("Olá, pessoal!", lang="pt")
voz.save("voz.mp3")

from IPython.display import Audio
Audio("voz.mp3")

In [ ]:
# Exibindo a lista de línguas
g.lang.tts_langs()

In [ ]:
arq = open("frases.txt", "r")
for linha in arq.readlines():
  lang, texto = linha.split(";")
  voz = g.gTTS(texto, lang=lang)
  voz.save("linha.mp3")
  display( Audio("linha.mp3") )

# Aula 5 - Speech to Text

In [ ]:
print("Dependências: veja README_LOCAL.md. Não é necessário apt no Windows.")

In [ ]:
import sounddevice as sd
import soundfile as sf
from IPython.display import HTML, Audio, display

def get_audio(duracao=5, sample_rate=44100):
    """Grava no microfone padrão do Windows durante duracao segundos."""
    print(f"Gravando por {duracao}s...")
    audio = sd.rec(int(duracao * sample_rate), samplerate=sample_rate,
                   channels=1, dtype="float32")
    sd.wait()
    return audio[:, 0], sample_rate


In [ ]:
# Obtém áudio do microfone, via navegador
audio, sr = get_audio()

In [ ]:
# Imports
import numpy as np
from scipy.io.wavfile import write
import speech_recognition as speechRecog

In [ ]:
def mic2wav(audio, wav_file, sample_rate=sr):
    # soundfile preserva a taxa real de captura e trata silêncio sem divisão por zero.
    sf.write(wav_file, audio, sample_rate, subtype="PCM_16")


In [ ]:
# Pede um áudio para o usuário
mic2wav(audio, "test3.wav")

In [ ]:
def transcreve_audio(wav_file):

  # Instancia o reconhecedor de voz
  recon = speechRecog.Recognizer()

  # Converte audio (.wav) para AudioFile
  audiofile = speechRecog.AudioFile(wav_file)

  # Converte AudioFile para AudioData
  with audiofile as source:
      audiodata = recon.record(audiofile)

  # Transcreve AudioData para texto
  text = recon.recognize_google(audiodata, language="pt")

  return text

In [ ]:
transcreve_audio("test3.wav")

In [ ]:
# Opcional: habilite para conversar com o microfone (requer internet para Google STT).
CONVERSAR = False
if CONVERSAR:
    for _ in range(20):
        audio, sr = get_audio()
        mic2wav(audio, "som.wav", sr)
        texto = transcreve_audio("som.wav")
        print("Você disse:", texto)
        if texto.strip().lower() == "pare":
            break
